In [ ]:
import pandas as pd
import json


def get_data():
    """
    select
        pl.id as player_id,
        pl.created_at as player_created_at,
        pl.possible_ban as player_possible_ban,
        pl.confirmed_ban as player_confirmed_ban,
        pl.confirmed_player as player_confirmed_player,
        pl.label_id as player_label_id,
        lb.label as player_label,
        case
            when label_jagex = 2 then "banned"
            else null
        end as player_account_status,
        hdl.scrape_date ,
        hdl.skills ,
        hdl.activities
    from Players pl
    join highscore_data_latest hdl on pl.id = hdl.player_id
    join Labels lb on pl.label_id = lb.id
    where 1=1
        and (pl.label_jagex = 2 or pl.confirmed_player = 1)
    order by pl.id asc
    ;
    """

    path = "../data/2025-08-30_17-21_training_data.csv.zip"
    df = pd.read_csv(filepath_or_buffer=path, sep=";", compression="zip")
    return df


df = get_data()
print(df.shape)
df.head(5)

In [ ]:
SKILLS = [
    "attack",
    "defence",
    "strength",
    "hitpoints",
    "ranged",
    "prayer",
    "magic",
    "cooking",
    "woodcutting",
    "fletching",
    "fishing",
    "firemaking",
    "crafting",
    "smithing",
    "mining",
    "herblore",
    "agility",
    "thieving",
    "slayer",
    "farming",
    "runecraft",
    "hunter",
    "construction",
]
MINIGAMES = [
    "lms_rank",
    "soul_wars_zeal",
    "cs_all",
    "cs_beginner",
    "cs_easy",
    "cs_medium",
    "cs_hard",
    "cs_elite",
    "cs_master",
    "pvp_arena_rank",
]
BOSSES = [
    "abyssal_sire",
    "alchemical_hydra",
    "barrows_chests",
    "bryophyta",
    "callisto",
    "cerberus",
    "chambers_of_xeric",
    "chambers_of_xeric_challenge_mode",
    "chaos_elemental",
    "chaos_fanatic",
    "commander_zilyana",
    "corporeal_beast",
    "crazy_archaeologist",
    "dagannoth_prime",
    "dagannoth_rex",
    "dagannoth_supreme",
    "deranged_archaeologist",
    "general_graardor",
    "giant_mole",
    "grotesque_guardians",
    "hespori",
    "kalphite_queen",
    "king_black_dragon",
    "kraken",
    "kreearra",
    "kril_tsutsaroth",
    "mimic",
    "nex",
    "nightmare",
    "phosanis_nightmare",
    "obor",
    "sarachnis",
    "scorpia",
    "skotizo",
    "tempoross",
    "the_gauntlet",
    "the_corrupted_gauntlet",
    "theatre_of_blood",
    "theatre_of_blood_hard",
    "thermonuclear_smoke_devil",
    "tombs_of_amascut",
    "tombs_of_amascut_expert",
    "tzkal_zuk",
    "tztok_jad",
    "venenatis",
    "vetion",
    "vorkath",
    "wintertodt",
    "zalcano",
    "zulrah",
    "rifts_closed",
    "calvarion",
    "the_whisperer",
    "scurrius",
    "artio",
    "vardorvis",
    "duke_sucellus",
    "the_leviathan",
    "phantom_muspah",
    "spindel",
    "yama",
    "araxxor",
    "amoxliatl",
    "sol_heredit",
    "lunar_chests",
    "the_hueycoatl",
    "colosseum_glory",
    "the_royal_titans",
    "doom_of_mokhaiotl",
    "collections_logged",
    "theatre_of_blood_hard_mode",
    "tombs_of_amascut_expert_mode",
]

FEATURE_COLUMNS = SKILLS + MINIGAMES + BOSSES

In [ ]:
skills_df = pd.DataFrame(
    [
        {k.lower(): v for k, v in json.loads(row).items()}
        for row in df.skills.values.tolist()
    ]
)
skills_df = skills_df.fillna(0)
skills_df.drop(columns=["overall"], inplace=True)
skills_df["total_skills"] = skills_df.sum(axis=1)

activities_df = pd.DataFrame(
    [
        {
            k.lower()
            .replace("-", " ")
            .replace("  ", "")
            .replace(" ", "_")
            .replace("'", "")
            .replace(":", ""): v
            for k, v in json.loads(row).items()
        }
        for row in df.activities.values.tolist()
    ]
)
activities_df = activities_df.fillna(0)


# Individual clue scroll categories
activities_df["cs_beginner"] = activities_df["clue_scrolls_(beginner)"]
activities_df["cs_easy"] = activities_df["clue_scrolls_(easy)"]
activities_df["cs_medium"] = activities_df["clue_scrolls_(medium)"]
activities_df["cs_hard"] = activities_df["clue_scrolls_(hard)"]
activities_df["cs_elite"] = activities_df["clue_scrolls_(elite)"]
activities_df["cs_master"] = activities_df["clue_scrolls_(master)"]

# Total clue scrolls
activities_df["cs_all"] = (
    activities_df["cs_beginner"]
    + activities_df["cs_easy"]
    + activities_df["cs_medium"]
    + activities_df["cs_hard"]
    + activities_df["cs_elite"]
    + activities_df["cs_master"]
)

cs_cols = [c for c in activities_df.columns if "clue_scrolls" in c]
bh_cols = [c for c in activities_df.columns if "bounty_hunter" in c]
activities_df = activities_df.drop(columns=cs_cols + bh_cols)

activities_df["total_minigames"] = activities_df[MINIGAMES].sum(axis=1)
activities_df["total_bosses"] = activities_df[BOSSES].sum(axis=1)
_ = [print(c) for c in activities_df.columns if c not in MINIGAMES + BOSSES]


In [ ]:
import numpy as np


def downcast_unsigned(df: pd.DataFrame) -> pd.DataFrame:
    """
    Downcast numeric columns to the smallest possible unsigned int type
    (uint8, uint16, uint32, uint64) based on column max values.
    NaN/inf should be handled before calling this.
    """
    # Unsigned integer dtypes and their limits
    uint_types = [np.uint8, np.uint16, np.uint32, np.uint64]

    for col in df.select_dtypes(include=[np.number]).columns:
        col_max = df[col].max()
        col_min = df[col].min()

        # Only downcast non-negative columns
        if col_min >= 0:
            for dtype in uint_types:
                if col_max <= np.iinfo(dtype).max:
                    df[col] = df[col].astype(dtype)
                    break
    return df

In [ ]:
df_concat = pd.concat([df, skills_df, activities_df], axis=1)
df_concat = df_concat.drop(columns=["skills", "activities"])
df_concat = downcast_unsigned(df=df_concat)
df_concat.head()

In [ ]:
# Remove players with no valid data
mask = (
    (df_concat["total_minigames"] == 0)
    & (df_concat["total_bosses"] == 0)
    & (df_concat["total_skills"] == 0)
)
print(df_concat[mask].shape)
df_concat = df_concat[~mask]

In [ ]:
columns = [
    "player_id",
    "player_possible_ban",
    "player_confirmed_ban",
    "player_label_jagex",
    "player_confirmed_player",
    "player_label_id",
    "player_label",
    "player_account_status",
    "player_created_at",
    "scrape_date",
    "total_skills",
    "total_minigames",
    "total_bosses",
]
_ = [print(c) for c in df_concat.columns if c not in FEATURE_COLUMNS + columns]

In [ ]:
import time
today_iso = time.strftime("%Y-%m-%d")
df_concat.to_parquet(f"../data/{today_iso}_hiscore_data.parquet.gzip", compression="gzip")